In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/holdings_2024_Q1.csv", low_memory=False)

# What percentage of unique CUSIPs are ETFs?
etf_keywords = ["ETF", "TRUST", "FUND", "INDEX", "ISHARES", 
                 "VANGUARD", "SPDR", "INVESCO", "BLACKROCK"]

mask = df["name"].str.upper().str.contains("|".join(etf_keywords), na=False)
print("ETF/Fund rows:", mask.sum())
print("Stock rows:", (~mask).sum())
print("ETF percentage:", f"{mask.sum()/len(df)*100:.1f}%")

# Show unique ETF-like names
etf_names = df[mask]["name"].unique()
print("\nSample ETF names:")
print(etf_names[:20])

ETF/Fund rows: 83453
Stock rows: 238205
ETF percentage: 25.9%

Sample ETF names:
<StringArray>
[         'SPDR S&P 500 ETF TR',   'SPDR SER TR SPDR BLOOMBERG',
      'MEDICAL PPTYS TRUST INC',                   'ISHARES TR',
                  'SPDR SER TR', 'VANGUARD INTL EQUITY INDEX F',
  'ABRDN JAPAN EQUITY FUND INC', 'BLACKROCK ENHANCED GLOBAL DI',
 'BLACKROCK CR ALLOCATION INCO', 'BLACKROCK HEALTH SCIENCES TR',
  'BLACKROCK ENHANCED INTL DIV',    'BLACKROCK MUNIVEST FD INC',
 'BLACKROCK ENHANCED GOVT FD I', 'BLACKROCK MUN TARGET TERM TR',
 'BLACKROCK HEALTH SCIENCES TE', 'BLACKROCK SCIENCE & TECHNOLO',
 'BLACKROCK INNOVATION AND GRW', 'BLACKROCK CAP ALLOCATION TER',
    'SRH TOTAL RETURN FUND INC',                'INVESCO BD FD']
Length: 20, dtype: str


In [2]:
import requests

HEADERS = {"User-Agent": "gvip-predictor bencheng18@gmail.com"}
response = requests.get(
    "https://www.sec.gov/Archives/edgar/data/1067983/000119312526226661/53405.xml",
    headers=HEADERS
)

from lxml import etree
root = etree.fromstring(response.content)
namespace = {"ns": "http://www.sec.gov/edgar/document/thirteenf/informationtable"}

for info in root.findall("ns:infoTable", namespace)[:5]:
    print({
        "name": info.findtext("ns:nameOfIssuer", namespaces=namespace),
        "titleOfClass": info.findtext("ns:titleOfClass", namespaces=namespace),
    })

{'name': 'ALLY FINL INC', 'titleOfClass': 'COM'}
{'name': 'ALLY FINL INC', 'titleOfClass': 'COM'}
{'name': 'ALLY FINL INC', 'titleOfClass': 'COM'}
{'name': 'ALLY FINL INC', 'titleOfClass': 'COM'}
{'name': 'ALLY FINL INC', 'titleOfClass': 'COM'}


In [3]:
# Use a wealth manager that likely holds ETFs
response = requests.get(
    "https://data.sec.gov/submissions/CIK0001602119.json",
    headers=HEADERS
)
data = response.json()

import pandas as pd
filings = pd.DataFrame(data["filings"]["recent"])
df_13f = filings[filings["form"] == "13F-HR"].iloc[0]

acc = df_13f["accessionNumber"].replace("-", "")
cik = 1602119
xml_url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/"

# Get the holdings XML
import requests, re
index_response = requests.get(
    f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/{df_13f['accessionNumber']}-index.htm",
    headers=HEADERS
)
xml_files = re.findall(r'href="([^"]+\.xml)"', index_response.text)
holdings_xml = [f for f in xml_files if "primary_doc" not in f]

if holdings_xml:
    filename = holdings_xml[0].split("/")[-1]
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/{filename}"
    response = requests.get(url, headers=HEADERS)
    root = etree.fromstring(response.content)
    namespace = {"ns": "http://www.sec.gov/edgar/document/thirteenf/informationtable"}
    
    seen = set()
    for info in root.findall("ns:infoTable", namespace):
        title = info.findtext("ns:titleOfClass", namespaces=namespace)
        name = info.findtext("ns:nameOfIssuer", namespaces=namespace)
        if title not in seen:
            print(f"{title}: {name}")
            seen.add(title)

COM CL A1: ACCEL ENTERTAINMENT INC
COM: CECO ENVIRONMENTAL CORP
COM SHS: CENTURI HOLDINGS INC
CL A COM: KURA SUSHI USA INC


In [1]:
import pandas as pd
import glob

files = sorted(glob.glob("../data/raw/holdings_*.csv"))

all_cusips = set()
for f in files:
    df = pd.read_csv(f, low_memory=False)
    all_cusips.update(df["cusip"].dropna().unique())

cusip_list = list(all_cusips)
print("Total CUSIPs:", len(cusip_list))

# Check for malformed CUSIPs
cusip_series = pd.Series(cusip_list)
print("\nCUSIP length distribution:")
print(cusip_series.str.len().value_counts().sort_index())

# Show examples of non-standard length CUSIPs
weird = cusip_series[cusip_series.str.len() != 9]
print("\nNon-standard CUSIPs:")
print(weird.values[:20])

Total CUSIPs: 27360

CUSIP length distribution:
9    27360
Name: count, dtype: int64

Non-standard CUSIPs:
<StringArray>
[]
Length: 0, dtype: str


In [1]:
import yfinance as yf

aapl = yf.Ticker("AAPL")

# Price history
hist = aapl.history(start="2020-01-01", end="2020-04-01")
print("Price history:")
print(hist.head())

# Fundamentals
info = aapl.info
keys = ["marketCap", "trailingPE", "priceToBook", "returnOnEquity", 
        "debtToEquity", "revenueGrowth", "grossMargins", "sector"]
print("\nFundamentals:")
for k in keys:
    print(f"  {k}: {info.get(k)}")

Price history:
                                Open       High        Low      Close  \
Date                                                                    
2020-01-02 00:00:00-05:00  71.344077  72.394108  71.091206  72.333900   
2020-01-03 00:00:00-05:00  71.563221  72.389273  71.406681  71.630653   
2020-01-06 00:00:00-05:00  70.754006  72.239935  70.503539  72.201401   
2020-01-07 00:00:00-05:00  72.211041  72.466322  71.642681  71.861839   
2020-01-08 00:00:00-05:00  71.565614  73.318870  71.565614  73.017830   

                              Volume  Dividends  Stock Splits  
Date                                                           
2020-01-02 00:00:00-05:00  135480400        0.0           0.0  
2020-01-03 00:00:00-05:00  146322800        0.0           0.0  
2020-01-06 00:00:00-05:00  118387200        0.0           0.0  
2020-01-07 00:00:00-05:00  108872000        0.0           0.0  
2020-01-08 00:00:00-05:00  132079200        0.0           0.0  

Fundamentals:
  marketCa

In [2]:
import yfinance as yf

aapl = yf.Ticker("AAPL")

# Historical quarterly financials
print(aapl.quarterly_income_stmt.T.head())
print(aapl.quarterly_balance_sheet.T.head())

            Tax Effect Of Unusual Items  Tax Rate For Calcs  \
2026-03-31                          0.0            0.175000   
2025-12-31                          0.0            0.175000   
2025-09-30                          0.0            0.162724   
2025-06-30                          0.0            0.164000   
2025-03-31                          0.0            0.154555   

            Normalized EBITDA  \
2026-03-31       3.932400e+10   
2025-12-31       5.406600e+10   
2025-09-30       3.555400e+10   
2025-06-30       3.103200e+10   
2025-03-31       3.225000e+10   

            Net Income From Continuing Operation Net Minority Interest  \
2026-03-31                                       2.957800e+10            
2025-12-31                                       4.209700e+10            
2025-09-30                                       2.746600e+10            
2025-06-30                                       2.343400e+10            
2025-03-31                                       2.4

In [3]:
import yfinance as yf

aapl = yf.Ticker("AAPL")

# Check available fields
print("Income statement fields:")
print(aapl.quarterly_income_stmt.index.tolist())

print("\nBalance sheet fields:")
print(aapl.quarterly_balance_sheet.index.tolist())

print("\nCash flow fields:")
print(aapl.quarterly_cashflow.index.tolist())

Income statement fields:
['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Normalized EBITDA', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Reconciled Cost Of Revenue', 'EBITDA', 'EBIT', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Total Expenses', 'Total Operating Income As Reported', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Net Income', 'Net Income Including Noncontrolling Interests', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Income Expense', 'Other Non Operating Income Expenses', 'Operating Income', 'Operating Expense', 'Research And Development', 'Selling General And Administration', 'Gross Profit', 'Cost Of Revenue', 'Total Revenue', 'Operating Revenue']

Balance sheet fields:
['Ordinary Shares Number', 'Share Issued', 'Net Debt', 'Total Debt', 'Tangible 

In [2]:
import pandas as pd
df = pd.read_csv("../data/processed/cusip_classifications.csv")
print(df.columns.tolist())
print(df[df["cusip"] == "037833100"])  # Apple

['cusip', 'security_type', 'security_type2', 'market_sector', 'name', 'ticker']
          cusip security_type security_type2 market_sector       name ticker
5649  037833100  Common Stock   Common Stock        Equity  APPLE INC   AAPL


In [3]:
import pandas as pd
df = pd.read_csv("../data/processed/crowding_features.csv")
print(df[df["cusip"] == "037833100"][["quarter_id", "total_holders", "top10_count"]].head(5))

     quarter_id  total_holders  top10_count
6015    2020_Q1            190           97
6016    2020_Q2            222          127
6017    2020_Q3            231          125
6018    2020_Q4            254          144
6019    2021_Q1            259          134


In [4]:
import pandas as pd

income = pd.read_csv("../data/processed/yfinance/financials/AAPL_income.csv", index_col=0)
print("Column names:")
print(income.columns.tolist()[:5])
print("\nColumn type:", type(income.columns[0]))

Column names:
['2026-03-31', '2025-12-31', '2025-09-30', '2025-06-30', '2025-03-31']

Column type: <class 'str'>


In [1]:
import pandas as pd
import glob
import os

files = sorted(glob.glob("../data/processed/yfinance/technical_*.csv"))

summary = []
for f in files:
    df = pd.read_csv(f)
    quarter = os.path.basename(f).replace("technical_", "").replace(".csv", "")
    summary.append({
        "quarter": quarter,
        "tickers": len(df),
        "return_3m_missing": df["return_3m"].isna().mean().round(3),
        "rsi_missing": df["rsi_14"].isna().mean().round(3),
        "macd_missing": df["macd_histogram"].isna().mean().round(3),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.sort_values("quarter").to_string(index=False))

quarter  tickers  return_3m_missing  rsi_missing  macd_missing
2020_Q1     2437              0.503        0.500         0.501
2020_Q2     2521              0.505        0.502         0.503
2020_Q3     2634              0.511        0.505         0.507
2020_Q4     2825              0.502        0.497         0.498
2021_Q1     2906              0.497        0.491         0.492
2021_Q2     3016              0.495        0.487         0.490
2021_Q3     3052              0.487        0.480         0.482
2021_Q4     3167              0.479        0.473         0.473
2022_Q1     3134              0.473        0.471         0.471
2022_Q2     3074              0.474        0.473         0.473
2022_Q3     3057              0.472        0.471         0.471
2022_Q4     3863              0.478        0.476         0.476
2023_Q1     3622              0.475        0.474         0.474
2023_Q2     3672              0.477        0.476         0.476
2023_Q3     3652              0.479        0.478       

In [1]:
import pandas as pd
df = pd.read_csv("../data/processed/yfinance/technical_2024_Q1.csv")
print("Rows:", len(df))
print("return_3m missing:", df["return_3m"].isna().mean().round(3))

Rows: 4131
return_3m missing: 0.136


In [2]:
import pandas as pd
df = pd.read_csv("../data/features/feature_matrix.csv")
print(df[["year", "quarter", "quarter_idx"]].drop_duplicates().sort_values("quarter_idx"))

    year  quarter  quarter_idx
1   2020        1            0
2   2020        2            1
23  2020        3            2
24  2020        4            3
25  2021        1            4
26  2021        2            5
3   2021        3            6
4   2021        4            7
5   2022        1            8
6   2022        2            9
7   2022        3           10
8   2022        4           11
9   2023        1           12
10  2023        2           13
11  2023        3           14
12  2023        4           15
0   2024        1           16
14  2024        2           17
15  2024        3           18
16  2024        4           19
17  2025        1           20
18  2025        2           21
19  2025        3           22
20  2025        4           23


In [1]:
import pandas as pd
df = pd.read_csv("../data/features/feature_matrix.csv")
print(df["quarter_idx"].dtype)
print(df["quarter_idx"].unique()[:5])

int64
[16  0  1  6  7]


In [5]:
import pandas as pd
import pickle
import os

df = pd.read_csv("../data/features/feature_matrix.csv")
model = pickle.load(open("../data/features/gvip_model.pkl", "rb"))

FEATURE_COLS = [
    "total_holders", "top10_count", "top10_ratio",
    "avg_portfolio_weight", "avg_rank", "total_value",
    "holders_qoq", "top10_qoq", "weight_qoq",
    "return_1m", "return_3m", "return_6m", "return_12m",
    "ma50_ratio", "ma200_ratio", "high52w_ratio",
    "volatility_3m", "volatility_ratio",
    "rel_volume", "dollar_volume",
    "rsi_14", "bollinger_position",
    "macd_histogram"
]

# Look at test set only
test = df[df["quarter_idx"] > 21].copy()
test["proba"] = model.predict_proba(test[FEATURE_COLS])[:, 1]

# Split predictions into:
# - Stocks that were ALREADY in top50 last quarter (persistence)
# - Stocks that are NEW entries this quarter
test["was_top50_last_quarter"] = test.groupby("cusip")["target"].shift(1).fillna(0)

# Among actual positives (target=1), what % were already top50?
positives = test[test["target"] == 1]
persistence_rate = positives["was_top50_last_quarter"].mean()
print(f"Of actual GVIP stocks, {persistence_rate:.1%} were already GVIP last quarter")

# Among new entries (wasn't top50 last quarter), how well do we predict?
new_entries = test[test["was_top50_last_quarter"] == 0]
from sklearn.metrics import roc_auc_score
if new_entries["target"].sum() > 0:
    auc_new = roc_auc_score(new_entries["target"], 
                             model.predict_proba(new_entries[FEATURE_COLS])[:, 1])
    print(f"AUC on NEW entries only: {auc_new:.4f}")

Of actual GVIP stocks, 44.0% were already GVIP last quarter
AUC on NEW entries only: 0.9986


In [10]:
import pandas as pd
import pickle
import os

PROJECT_ROOT = "/Users/for_everyoung10/Documents/gvip-predictor"

df = pd.read_csv(f"{PROJECT_ROOT}/data/features/feature_matrix.csv", low_memory=False)
ticker_map = pd.read_csv(f"{PROJECT_ROOT}/data/processed/cusip_ticker_map.csv")
model = pickle.load(open(f"{PROJECT_ROOT}/data/features/gvip_model.pkl", "rb"))

FEATURE_COLS = [
    "total_holders", "top10_count", "top10_ratio",
    "avg_portfolio_weight", "avg_rank", "total_value",
    "holders_qoq", "top10_qoq", "weight_qoq",
    "return_1m", "return_3m", "return_6m", "return_12m",
    "ma50_ratio", "ma200_ratio", "high52w_ratio",
    "volatility_3m", "volatility_ratio",
    "rel_volume", "dollar_volume",
    "rsi_14", "bollinger_position",
    "macd_histogram"
]

# Get 2025 Q3 data (quarter_idx = 22)
q3 = df[df["quarter_idx"] == 22].copy()
q3["proba"] = model.predict_proba(q3[FEATURE_COLS])[:, 1]

# Get 2025 Q4 data for actual GVIP
q4 = df[df["quarter_idx"] == 23].copy()

# Top 50 predicted vs actual
top50_pred = set(q3.nlargest(50, "proba")["cusip"].tolist())
top50_actual = set(q4.nlargest(50, "top10_count")["cusip"].tolist())

# Overlap
overlap = top50_pred & top50_actual
missed = top50_actual - top50_pred
false_positives = top50_pred - top50_actual

print(f"Overlap: {len(overlap)}/50")
print(f"Missed: {len(missed)}")
print(f"False positives: {len(false_positives)}")

# Missed stocks — what did they look like in Q3?
missed_q3 = q3[q3["cusip"].isin(missed)][
    ["cusip", "top10_count", "total_holders", "proba", "holders_qoq", "top10_qoq"]
].copy().merge(ticker_map, on="cusip", how="left")

print("\nMissed stocks (in actual GVIP Q4 but not predicted from Q3):")
print(missed_q3.sort_values("proba", ascending=False).to_string(index=False))

# False positives — predicted but not in actual GVIP
fp_q3 = q3[q3["cusip"].isin(false_positives)][
    ["cusip", "top10_count", "total_holders", "proba", "holders_qoq", "top10_qoq"]
].copy().merge(ticker_map, on="cusip", how="left")

print("\nFalse positives (predicted but not in actual GVIP Q4):")
print(fp_q3.sort_values("proba", ascending=False).to_string(index=False))

Overlap: 43/50
Missed: 7
False positives: 7

Missed stocks (in actual GVIP Q4 but not predicted from Q3):
    cusip  top10_count  total_holders    proba  holders_qoq  top10_qoq ticker
191216100          102           1211 0.982366         59.0      -11.0     KO
949746101          103            794 0.980150         79.0      -14.0    WFC
512807306           81            478 0.924053        112.0       18.0   LRCX
038222105           78            637 0.894349         62.0       18.0   AMAT
595112103           71            488 0.891274         86.0       15.0     MU
58933Y105           77           1165 0.874944        112.0        9.0    MRK
031162100           62            907 0.850469         45.0       -1.0   AMGN

False positives (predicted but not in actual GVIP Q4):
    cusip  top10_count  total_holders    proba  holders_qoq  top10_qoq ticker
040413205           79            432 0.996110        103.0       31.0   ANET
697435105           94            708 0.996043         79.

In [11]:
print(ticker_map[ticker_map["ticker"] == "MLB1"])

          cusip ticker
4384  58733R102   MLB1
